---
title: "Core Module: Internal functions and testing"
exec_all: true
---

## core

> This is a core library for the ERA5 dataset pipeline. It defines a few helpful functions such as an API tester to test your API key and connection.

<!-- WARNING: THIS FILE WAS AUTOGENERATED! DO NOT EDIT! -->

In [ ]:
#| code-fold: show
#| code-summary: "Exported source"
#| exports: #
import os
import cdsapi
import hydra
import json
import tempfile
import argparse
import zipfile
import shutil
import geopandas as gpd
from pathlib import Path
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
from omegaconf import DictConfig, OmegaConf
from pyprojroot import here
from importlib import import_module

## Utilities

Some utilities are provided to help you with the ERA5 dataset.

In [0]:
#| echo: false
#| output: asis
show_doc(describe)

---

[source](https://github.com/TinasheMTapera/era5_sandbox/blob/main/era5_sandbox/core.py#L26){target="_blank" style="float:right; font-size:smaller"}

### describe

>      describe (cfg:omegaconf.dictconfig.DictConfig=None)

*Describe the configuration file used by Hydra for the pipeline*

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| cfg | DictConfig | None | Configuration file |
| **Returns** | **None** |  |  |

In [ ]:
#| code-fold: show
#| code-summary: "Exported source"
#| exports: #
def describe(
    cfg: DictConfig=None,  # Configuration file
    )-> None:
    "Describe the configuration file used by Hydra for the pipeline"
    
    if cfg is None:
        print("No configuration file provided. Generating default configuration file.")
        cfg = OmegaConf.create()
        
    print("This package fetches ERA5 data. The following is the config file used by Hydra for the pipeline:\n")
    print(OmegaConf.to_yaml(cfg))

In addition, we've defined 3 private functions to help with path expansion [`_expand_path`](https://TinasheMTapera.github.io/era5_sandbox/core.html#_expand_path), dynamic function importing [`_get_callable`](https://TinasheMTapera.github.io/era5_sandbox/core.html#_get_callable), and directory structure creation [`_create_directory_structure`](https://TinasheMTapera.github.io/era5_sandbox/core.html#_create_directory_structure).

### A Simple Temperature Conversion Function

In [0]:
#| echo: false
#| output: asis
show_doc(kelvin_to_celsius)

---

[source](https://github.com/TinasheMTapera/era5_sandbox/blob/main/era5_sandbox/core.py#L81){target="_blank" style="float:right; font-size:smaller"}

### kelvin_to_celsius

>      kelvin_to_celsius (kelvin:float)

*Convert temperature from Kelvin to Celsius.*

|    | **Type** | **Details** |
| -- | -------- | ----------- |
| kelvin | float | Temperature in Kelvin |
| **Returns** | **float** | **Temperature in Celsius** |

### A Class for Authenticating Google Drive

We're going to use a class to authenticate and interact with google drive. The goal is to have a simple interface to fetch the healthshed files dynamically from google drive in the pipeline.

::: {.callout-important}
This class was implemented when all of our data
was stored on a private Google Drive. Since we
have moved all of our data to FASRC, this will
likely be deprecated in the near future.
:::

In [0]:
#| echo: false
#| output: asis
show_doc(GoogleDriver)

---

[source](https://github.com/TinasheMTapera/era5_sandbox/blob/main/era5_sandbox/core.py#L91){target="_blank" style="float:right; font-size:smaller"}

### GoogleDriver

>      GoogleDriver (json_key_path=None)

*A class to handle Google Drive authentication and file management.
This class uses the PyDrive2 library to authenticate with Google Drive using a service account.

It provides three methods: authenticating the account, getting the drive object, and downloading the healthshed files for madagascar.*

Here's how we use it. The credentials for the data-pipeline service account are
available in the sandbox folder, and the path to said folder is set in the config:

In [ ]:
from hydra import initialize, compose
from omegaconf import OmegaConf

In [ ]:
# unfortunately, we have to use the initialize function to load the config file
# this is because the @hydra decorator does not work with Notebooks very well
# this is a known issue with Hydra: https://gist.github.com/bdsaglam/586704a98336a0cf0a65a6e7c247d248
# 
# just use the relative path from the notebook to the config dir
try:
    with initialize(version_base=None, config_path="../conf"):
        cfg = compose(config_name='config.yaml')
except Exception as e:
    print(f"Error initializing Hydra: {e}")
    with initialize(version_base=None, config_path="conf"):
        cfg = compose(config_name='config.yaml')

::: {.callout-important}
If we continue with `pytask`, we will not need to
use hydra at all, and so the above strategy
may get deprecated.
:::

In [ ]:
auth = GoogleDriver(json_key_path=here() / cfg.GOOGLE_DRIVE_AUTH_JSON.path)
drive = auth.get_drive()

Here's how we might check that the healthsheds are accessible in the drive:

In [ ]:
# we're using the madagascar healthshed folder as an example
folder_id = cfg.geographies.madagascar.healthsheds
folder_name = "healthsheds2022.zip"
file_list = drive.ListFile({'q': f" title='{folder_name}' and trashed = false "}).GetList()

for file in file_list:
    print(f"{file['title']} - {file['mimeType']}")

That being said, we can read in  the healthsheds into geopandas by downloading them to a temp directory. The healthsheds must be a zipped shapefiles package with the files at the root of the zip directory.

In [ ]:
with tempfile.TemporaryDirectory() as temp_dir:
    # Create a temporary directory to store the downloaded file
    zip_path = os.path.join(temp_dir, folder_name)

    # Download file from Google Drive
    file_obj = drive.CreateFile({'id': file_list[0]['id']})
    file_obj.GetContentFile(zip_path)

    # Read shapefile directly from ZIP
    gdf = gpd.read_file(f"zip://{zip_path}")

That works! So now we can patch the class to include this workflow:

In [0]:
#| echo: false
#| output: asis
show_doc(GoogleDriver.read_healthsheds)

---

[source](https://github.com/TinasheMTapera/era5_sandbox/blob/main/era5_sandbox/core.py#L128){target="_blank" style="float:right; font-size:smaller"}

### GoogleDriver.read_healthsheds

>      GoogleDriver.read_healthsheds (healthshed_zip_name)

And to check that it works:

In [ ]:
driver = GoogleDriver(json_key_path=here() / cfg.GOOGLE_DRIVE_AUTH_JSON.path)
drive = driver.get_drive()
healthsheds = driver.read_healthsheds("healthsheds2022.zip")

healthsheds.describe()

## CDS File Handler Type

::: {.callout-important}
This section may also be deprecated. Since adding `swvl1` to the pipeline, we have not needed to use this class. We leave it here for now for reference.
:::

We're going to make a file handler type to help deal with CDS files. This is to fix [NSAPH-Data-Processing/era5_sandbox#13](https://github.com/NSAPH-Data-Processing/era5_sandbox/issues/13). 

Usually, when you download data, it comes out as a simple .nc file that can be opened with xarray. However, the CDS API has a few different file types that are not .nc files. For example, the ERA5 data is stored in a .grib file format. This is a common format for meteorological data, and it is used by the ECMWF. When a query has multiple variables, sometimes they are downloaded as a .zip file to separat the grib from the netcdf.

So, below, we define a class that can handle the file no matter what the type is. It will check the file type and then use the appropriate method to open it. The class will also have a method to check if the file is a .zip file, and if so, it will unzip it and return the path to the unzipped file.

In [0]:
#| echo: false
#| output: asis
show_doc(ClimateDataFileHandler)

---

[source](https://github.com/TinasheMTapera/era5_sandbox/blob/main/era5_sandbox/core.py#L151){target="_blank" style="float:right; font-size:smaller"}

### ClimateDataFileHandler

>      ClimateDataFileHandler (input_path:str)

*A class to handle file operations for the Climate Data Store (CDS).
This class provides unpack files downloaded from the CDS API. It must be able to
handle the unpacking of files downloaded from the CDS API. This means that
if the file is a basic netcdf, it should be passed to the netcdf handler. If
the file is a zip, it should be handled by the zip handler in temp and the
data returned as required.*

In [ ]:
import xarray as xr
from fastcore.test import test_fail

In [ ]:
eg_file = here() / "bld/2019_5_madagascar.nc"

# this fails because the nc file downloaded has grib and netcdf in it, so
# xr cannot handle it.
def wont_work(multilayer_file):

    ds = xr.open_dataset(multilayer_file)

test_fail(
    wont_work,
    args=(eg_file)
)

# equivalent to saying try: wont_work(eg_file) Except: some error handling

The above fails because the download contains temperature and precipitation data, which get encoded silently as different formats. Even though it is one file, it contains both grib and netcdf data and is encoded as a .zip file. So we use the class to read it instead:

In [ ]:
handler = ClimateDataFileHandler(eg_file)
handler.prepare()
ds1 = xr.open_dataset(handler.get_dataset("instant"))
#ds2 = xr.open_dataset(handler.get_dataset("accum"))

::: {.callout-important}
The above line for `ds2` is commented out because the example file does not separate accumulation data. 
:::

In [ ]:
ds1

In [ ]:
#ds2

In [ ]:
handler.cleanup()

Great! Let's add a context handler and this can be added to the pipeline,
so that with the entry and exit methods, we can now use the class in a `with` statement:

In [ ]:
with ClimateDataFileHandler(eg_file) as handler:
    ds1 = xr.open_dataset(handler.get_dataset("instant"))
    #ds2 = xr.open_dataset(handler.get_dataset("accum"))

    print(ds1)
    #print(ds2)

## Tests and Main

In `nbdev`, our tests are embedded in the notebook. Whenever you export the notebook, all the cells that are specified to run are run, and hence, the tests are executed. The tests are also exported. This is a great way to ensure that your documentation is always up-to-date. For this module, we're using the [`testAPI()`](https://TinasheMTapera.github.io/era5_sandbox/core.html#testapi) function as our main test.

In [0]:
#| echo: false
#| output: asis
show_doc(testAPI)

---

[source](https://github.com/TinasheMTapera/era5_sandbox/blob/main/era5_sandbox/core.py#L254){target="_blank" style="float:right; font-size:smaller"}

### testAPI

>      testAPI (cfg:omegaconf.dictconfig.DictConfig=None,
>               dataset:str='reanalysis-era5-pressure-levels')

In [ ]:
#| code-fold: show
#| code-summary: "Exported source"
#| exports: #
def testAPI(
    cfg: DictConfig=None,
    dataset:str="reanalysis-era5-pressure-levels"
    )-> bool:    
    
    # parse config
    testing=cfg.development_mode
    output_path=here("data") / "testing"

    print(OmegaConf.to_yaml(cfg))

    try:
        client = cdsapi.Client()

        # build request
        request = {
            'product_type': ['reanalysis'],
            'variable': ['geopotential'],
            'year': ['2024'],
            'month': ['03'],
            'day': ['01'],
            'time': ['13:00'],
            'pressure_level': ['1000'],
            'data_format': 'grib',
        }

        target = output_path / 'test_download.grib'
        
        print("Testing API connection by downloading a dummy dataset to {}...".format(output_path))

        client.retrieve(dataset, request, target)

        if not testing:
            os.remove(target)
        
        print("API connection test successful.")
        return True

    except Exception as e:
        print("API connection test failed.")
        print("Did you set up your API key with CDS? If not, please visit https://cds.climate.copernicus.eu/how-to-api#install-the-cds-api-client")
        print("Error: {}".format(e))
        return False

We can see that this API tester tool works with Hydra configuration:

In [ ]:
from hydra import initialize, compose
from omegaconf import OmegaConf

In [ ]:
# unfortunately, we have to use the initialize function to load the config file
# this is because the @hydra decorator does not work with Notebooks very well
# this is a known issue with Hydra: https://gist.github.com/bdsaglam/586704a98336a0cf0a65a6e7c247d248
# 
# just use the relative path from the notebook to the config dir
try:
    with initialize(version_base=None, config_path="../conf"):
        cfg = compose(config_name='config.yaml')
except Exception as e:
    print(f"Error initializing Hydra: {e}")
    with initialize(version_base=None, config_path="conf"):
        cfg = compose(config_name='config.yaml')

describe(cfg)

### Importing the Main Function

::: {.callout-important}
As mentioned, if we continue with `pytask`, we will not need to use hydra at all, and so the main function
may get deprecated as `pytask` will handle the pipeline execution without `__main__` scripts.
:::

Important: using `__main__` in nbdev and Hydra is a little bit tricky. We need to define the main function in the module ONLY ONCE and then when we export the notebook to script, we need to add the `nbdev.imports.IN_NOTEBOOK` variable. This way, the main function will only be executed when we run the notebook and not when we import the module.

```python
from nbdev.imports import IN_NOTEBOOK
```

You'll see this listed throughout the notebooks.

In [0]:
#| echo: false
#| output: asis
show_doc(main)

---

[source](https://github.com/TinasheMTapera/era5_sandbox/blob/main/era5_sandbox/aggregate.py#L302){target="_blank" style="float:right; font-size:smaller"}

### main

>      main (cfg:omegaconf.dictconfig.DictConfig)

In [ ]:
#| code-fold: show
#| code-summary: "Exported source"
#| exports: #
@hydra.main(version_base=None, config_path="../../conf", config_name="config")
def main(cfg: DictConfig) -> None:

    # Create the directory structure
    _create_directory_structure(here() / "data", cfg.datapaths)

    # test the api
    testAPI(cfg=cfg)

In [ ]:
#| export: null
#| eval: false
try: from nbdev.imports import IN_NOTEBOOK
except: IN_NOTEBOOK=False

if __name__ == "__main__" and not IN_NOTEBOOK:
    main()